# Day 5 · 관측·평가·배포 운영

하나의 입력을 8개 차시 동안 확장합니다. 웹사이트 확인이 아니라 코드·명령·test·결과 파일을 직접 다루며, 모든 외부 쓰기는 dry-run과 사람 승인을 먼저 거칩니다.

In [ ]:
from pathlib import Path
import importlib.util, json, subprocess, sys

def find_workspace(start):
    for candidate in [start, *start.parents]:
        if (candidate / "requirements-day1.txt").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("WORKSPACE_ROOT_NOT_FOUND")

ROOT = find_workspace(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({"workspace": ROOT.name, "python": sys.version.split()[0]})

In [ ]:
# 최초 1회. 없는 핵심 library가 있을 때만 현재 Notebook Kernel에 설치합니다.
required = ["pydantic", "pytest", "langchain_core", "langgraph"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-day1.txt")],
        check=True,
    )
print({"missing_before_install": missing, "environment_ready": True})

# 실제 STT를 실행할 사람만 requirements-stt-optional.txt를 별도로 설치합니다.

In [ ]:
OUT = ROOT / "output/course-labs/day5"
OUT.mkdir(parents=True, exist_ok=True)

def save_json(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print({"saved": str(path.relative_to(ROOT))})
    return path

def run_command(*args, cwd=ROOT):
    completed = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    display_args = list(args)
    if display_args and display_args[0] == sys.executable:
        display_args[0] = "python"
    result = {
        "command": " ".join(display_args),
        "returncode": completed.returncode,
        "stdout_tail": completed.stdout.strip().splitlines()[-5:],
        "stderr_tail": completed.stderr.strip().splitlines()[-5:],
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result

## 1차시 · Meeting·Review Service Router

모델이 업무 종류와 권한을 추측하지 않도록 호출자가 `input_kind`를 명시합니다.

In [ ]:
from src.course_services.service_router import route_service_request

meeting = route_service_request(
    input_kind="meeting_transcript", source_path=ROOT / "data/meeting_sample_ko.txt", workspace_root=ROOT,
)
review = route_service_request(
    input_kind="code_diff", source_path=ROOT / "data/day3_review_cases/unsafe_pr.diff", workspace_root=ROOT,
)
unknown = route_service_request(
    input_kind="unknown", source_path=ROOT / "data/meeting_sample_ko.txt", workspace_root=ROOT,
)
router_result = {"meeting": meeting, "review": review, "boundary": unknown}
assert unknown["error_code"] == "UNSUPPORTED_INPUT_KIND"
save_json("01_unified_service_result.json", router_result)
{"meeting": meeting["service"], "review": review["service"], "boundary": unknown["error_code"]}

## 2차시 · Local Trace와 LangSmith 선택 Upload

기본은 local JSON입니다. synthetic·deidentified 결과 요약만 사람 선택 뒤 LangSmith에 올립니다.

In [ ]:
from src.observability_lab import LocalTraceRecorder

trace_recorder = LocalTraceRecorder(
    run_name="day5-agent-operations",
    metadata={"data_classification": "synthetic", "external_write": False},
)
with trace_recorder.span("route_meeting", inputs={"input_kind": "meeting_transcript"}):
    assert meeting["status"] == "SUCCESS"
with trace_recorder.span("route_code_review", inputs={"input_kind": "code_diff"}):
    assert review["status"] == "SUCCESS"
trace = trace_recorder.to_dict()
trace["langsmith_upload"] = {"requested": False, "reason": "LOCAL_FIRST"}
save_json("02_trace.json", trace)
[(span["name"], span["status"]) for span in trace["spans"]]

## 3차시 · 운영 Metadata와 Monitoring

평균 하나가 아니라 provider·status·latency·fallback·READY/HOLD를 함께 봅니다.

In [ ]:
monitoring_view = {
    "run_name": trace["run_name"],
    "status_counts": {
        "SUCCESS": sum(span["status"] == "SUCCESS" for span in trace["spans"]),
        "ERROR": sum(span["status"] == "ERROR" for span in trace["spans"]),
    },
    "latency_ms": {span["name"]: span["latency_ms"] for span in trace["spans"]},
    "filters": ["provider_used", "error_code", "fallback_reason", "decision"],
    "raw_content_uploaded": False,
}
save_json("03_monitoring_view.json", monitoring_view)
monitoring_view

## 4차시 · Dataset Experiment와 Release Gate

같은 Golden Set으로 baseline과 candidate를 비교하고 threshold 아래면 HOLD합니다.

In [ ]:
from src.course_services.eval_service import evaluate_review_findings, release_gate

expected = json.loads((ROOT / "data/day5_eval/golden_review_findings.json").read_text(encoding="utf-8"))
experiment_metrics = evaluate_review_findings(review["result"]["findings"], expected)
ready_gate = release_gate(review_metrics=experiment_metrics, safety_passed=True, latency_seconds=0.2)
hold_gate = release_gate(review_metrics={"precision": 0.7, "recall": 0.6}, safety_passed=False, latency_seconds=42.0)
assert ready_gate["decision"] == "READY" and hold_gate["decision"] == "HOLD"
save_json("04_experiment_result.json", {"metrics": experiment_metrics, "normal": ready_gate, "boundary": hold_gate})
{"metrics": experiment_metrics, "normal": ready_gate["decision"], "boundary": hold_gate["decision"]}

## 5차시 · Human Feedback과 Annotation

좋아요 한 개가 아니라 approve·edit·reject와 판단 이유를 구조화합니다.

In [ ]:
feedback = [
    {"run_id": "synthetic-001", "decision": "approve", "reason": "EVIDENCE_CONFIRMED", "reviewer": "human"},
    {"run_id": "synthetic-002", "decision": "edit", "reason": "OWNER_CORRECTED", "reviewer": "human"},
    {"run_id": "synthetic-003", "decision": "reject", "reason": "UNKNOWN_EVIDENCE", "reviewer": "human"},
]
feedback_path = OUT / "05_human_feedback.jsonl"
feedback_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in feedback), encoding="utf-8")
print({"saved": str(feedback_path.relative_to(ROOT)), "rows": len(feedback)})
feedback

## 6차시 · PII·Retention·Incident 운영

외부 trace 전에 이메일·전화번호·token 형태를 가리고, raw content를 업로드하지 않습니다.

In [ ]:
from src.observability_lab import redact_observability_text

sample_sensitive = "담당자 test@example.com 010-1234-5678 sk-exampletoken123456789"
redaction = redact_observability_text(sample_sensitive)
ops_checklist = {
    "classification": "synthetic", "redaction": redaction, "retention_days": 7,
    "raw_content_upload": False, "incident_owner_required": True,
}
assert redaction["redacted"] is True
assert "test@example.com" not in redaction["text"]
save_json("06_ops_checklist.json", ops_checklist)
ops_checklist

## 7차시 · Release Candidate와 Local Web Demo

Python 결과와 브라우저 Demo가 같은 JSON을 사용하도록 먼저 Demo data를 생성하고, Node가 있으면 local build를 실행합니다.

In [ ]:
import shutil
from src.course_services.course_demo import write_course_demo

release_data = {
    day: write_course_demo(day, workspace_root=ROOT, output_dir=ROOT / f"output/course-demos/day{day}")
    for day in (2, 3, 4, 5)
}
node = shutil.which("node")
web_build = run_command(node, "build.mjs", cwd=ROOT / "web-demo") if node else {
    "command": "node build.mjs", "returncode": 2, "error_code": "NODE_NOT_AVAILABLE"
}
release_candidate = {
    "demo_decisions": {str(day): result["decision"] for day, result in release_data.items()},
    "web_build": web_build,
    "schema_source": "output/course-demos/dayN/demo_result.json",
}
save_json("07_release_candidate.json", release_candidate)
release_candidate

## 8차시 · 최종 Demo와 Scorecard

정상 한 건과 대표 HOLD 한 건, 사람 승인, 평가 수치를 함께 보여주고 전체 test로 마무리합니다.

In [ ]:
from src.course_services.course_demo import build_course_demo

final_scorecard = build_course_demo(5, workspace_root=ROOT)
focused_test = run_command(sys.executable, "-m", "pytest", "-q", "tests/test_course_services.py", "tests/test_observability_workflow.py")
final_scorecard["focused_test"] = focused_test
final_scorecard["web_build"] = web_build
assert final_scorecard["decision"] == "READY"
assert final_scorecard["boundary_case"]["status"] == "HOLD"
assert focused_test["returncode"] == 0
save_json("08_release_scorecard.json", final_scorecard)
{"decision": final_scorecard["decision"], "boundary": final_scorecard["boundary_case"], "metrics": final_scorecard["metrics"]}

## 완료 확인

- Day 5의 1~8차시 결과 파일을 확인했습니다.
- 정상 경로와 가장 중요한 실패 경로를 모두 실행했습니다.
- 외부 쓰기와 자동 메일이 기본값 `false`임을 확인했습니다.
- Codex·Claude Code 결과는 test와 diff를 사람이 검토한 뒤에만 반영합니다.